In [1]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

In [2]:
X = pd.read_csv('credit_card_featured.csv')

In [3]:
raw_cols = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
       'SEP_REPAY_STATUS', 'AUG_REPAY_STATUS', 'JUL_REPAY_STATUS',
       'JUN_REPAY_STATUS', 'MAY_REPAY_STATUS', 'APR_REPAY_STATUS',
       'SEP_BILL_AMT', 'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT',
       'MAY_BILL_AMT', 'APR_BILL_AMT', 'SEP_PAY_AMT', 'AUG_PAY_AMT',
       'JUL_PAY_AMT', 'JUN_PAY_AMT', 'MAY_PAY_AMT', 'APR_PAY_AMT']
payment_ratio_cols = ['PAY_RATIO_SEP', 'PAY_RATIO_AUG',
       'PAY_RATIO_JUL', 'PAY_RATIO_JUN', 'PAY_RATIO_MAY']
util_rate_cols = ['UTIL_SEP', 'UTIL_AUG', 'UTIL_JUL', 'UTIL_JUN', 'UTIL_MAY', 'UTIL_APR']
aggregate_cols = ['MONTHS_NO_BALANCE', 'BILL_GROWTH_RATIO', 'UTIL_TREND', 'PAY_TREND', 
       'UTIL_AVG', 'UTIL_MAX', 'UTIL_MIN', 'UTIL_STD']

### Payment ratios

In [4]:
#X = X.select_dtypes(include='number')
y = X['default.payment.next.month']
#X = X.drop(columns=['default.payment.next.month'])

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

In [7]:
model = XGBClassifier(eval_metric = 'logloss', random_state=0)

### Cumulative 

In [8]:
sets = {
    'raw': raw_cols,
    '+ payment ratios': raw_cols + payment_ratio_cols,
    '+ utilization': raw_cols + payment_ratio_cols + util_rate_cols,
    '+ aggregates': raw_cols + payment_ratio_cols + util_rate_cols + aggregate_cols,
}

for name, cols in sets.items():
    auc = cross_val_score(model, X_train[cols], y_train, cv=cv, scoring='roc_auc')
    print('%s: AUC %.4f +/- %.4f' % (name, auc.mean(), auc.std()))

raw: AUC 0.7610 +/- 0.0068


+ payment ratios: AUC 0.7585 +/- 0.0058


+ utilization: AUC 0.7606 +/- 0.0067


+ aggregates: AUC 0.7570 +/- 0.0053


### Individual

In [9]:
sets = {
    'raw': raw_cols,
    '+ payment ratios only': raw_cols + payment_ratio_cols,
    '+ utilization only': raw_cols + util_rate_cols,
    '+ aggregates only': raw_cols + aggregate_cols,
}

for name, cols in sets.items():
    auc = cross_val_score(model, X_train[cols], y_train, cv=cv, scoring='roc_auc')
    print('%s: AUC %.4f +/- %.4f' % (name, auc.mean(), auc.std()))

raw: AUC 0.7610 +/- 0.0068


+ payment ratios only: AUC 0.7585 +/- 0.0058


+ utilization only: AUC 0.7594 +/- 0.0061


+ aggregates only: AUC 0.7590 +/- 0.0064


### Engineered features without raw features

In [10]:
sets = {
    'payment ratios only': payment_ratio_cols,
    'utilization only': util_rate_cols,
    'aggregates only': aggregate_cols,
    'all together': payment_ratio_cols + util_rate_cols + aggregate_cols,
    'payment ratios and aggregates': payment_ratio_cols + aggregate_cols
}

for name, cols in sets.items():
    auc = cross_val_score(model, X_train[cols], y_train, cv=cv, scoring='roc_auc')
    print('%s: AUC %.4f +/- %.4f' % (name, auc.mean(), auc.std()))

payment ratios only: AUC 0.6764 +/- 0.0079


utilization only: AUC 0.6209 +/- 0.0074


aggregates only: AUC 0.6407 +/- 0.0091


all together: AUC 0.6919 +/- 0.0062


payment ratios and aggregates: AUC 0.6923 +/- 0.0061
